# 03. Recursive Chunking 실험

재귀적 청킹 전략 실험 및 평가

## 1. 환경 설정

In [ ]:
import sys
sys.path.append('..')

import json
import yaml
import time
from pathlib import Path

from src.chunkers import RecursiveChunker
from src.embedders import BGEEmbedder
from src.retrievers import VectorRetriever
from src.utils.document_loader import DocumentLoader

## 2. 설정 로드

In [ ]:
# Recursive chunking 설정 로드
with open('../configs/recursive_config.yaml', 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

print("Recursive Chunking 설정:")
print(json.dumps(config['chunking'], indent=2, ensure_ascii=False))

## 3. 데이터 로드

In [ ]:
# 전처리된 문서 로드
loader = DocumentLoader()
documents = loader.load_processed('../data/processed/cleaned_documents.json')

print(f"총 {len(documents)}개 문서 로드됨")

## 4. Recursive Chunking 실행

In [ ]:
# Chunker 초기화
chunker = RecursiveChunker(config['chunking'])
print(f"Chunker: {chunker}")
print(f"\n구분자 우선순위:")
for i, sep in enumerate(chunker.separators, 1):
    print(f"  {i}. {repr(sep)}")

In [ ]:
# 청킹 실행
print("\nRecursive Chunking 실행 중...")
print("=" * 60)

start_time = time.time()
chunks = chunker.chunk_documents(documents, verbose=True)
chunking_time = time.time() - start_time

print(f"\n총 {len(chunks)}개 청크 생성 (소요 시간: {chunking_time:.2f}초)")

## 5. 청크 통계

In [ ]:
# 청크 통계
stats = chunker.get_statistics(chunks)

print("청크 통계:")
print("=" * 40)
for key, value in stats.items():
    if isinstance(value, float):
        print(f"{key}: {value:.2f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# 청크 크기 분포 시각화
import matplotlib.pyplot as plt

sizes = [chunk.word_count for chunk in chunks]

plt.figure(figsize=(10, 4))
plt.hist(sizes, bins=30, color='darkorange', edgecolor='black', alpha=0.7)
plt.axvline(stats['avg_size'], color='red', linestyle='--', label=f"Mean: {stats['avg_size']:.0f}")
plt.axvline(stats['median_size'], color='green', linestyle=':', label=f"Median: {stats['median_size']:.0f}")
plt.xlabel('Chunk Size (words)')
plt.ylabel('Frequency')
plt.title('Recursive Chunking - Chunk Size Distribution')
plt.legend()
plt.tight_layout()
plt.show()

## 6. 샘플 청크 확인

Recursive Chunking은 문서 구조를 보존하므로 섹션 단위로 깔끔하게 분리됩니다.

In [ ]:
# 샘플 청크 확인
print("샘플 청크 (처음 3개):")
print("=" * 60)

for chunk in chunks[:3]:
    print(f"\n[{chunk.chunk_id}] ({chunk.word_count} words)")
    print("-" * 40)
    print(chunk.text[:400] + "..." if len(chunk.text) > 400 else chunk.text)

## 7. Fixed vs Recursive 비교

In [ ]:
# Fixed chunking 결과 로드 (이전 노트북에서 생성)
try:
    with open('../results/fixed_chunking/stats.json', 'r') as f:
        fixed_stats = json.load(f)
    
    print("Fixed vs Recursive 비교:")
    print("=" * 50)
    print(f"{'메트릭':<20} {'Fixed':<15} {'Recursive':<15}")
    print("-" * 50)
    print(f"{'총 청크 수':<20} {fixed_stats['total_chunks']:<15} {stats['total_chunks']:<15}")
    print(f"{'평균 크기':<20} {fixed_stats['avg_size']:<15.1f} {stats['avg_size']:<15.1f}")
    print(f"{'표준편차':<20} {fixed_stats['std_size']:<15.1f} {stats['std_size']:<15.1f}")
    print(f"{'크기 균일성':<20} {fixed_stats['size_uniformity']:<15.2f} {stats['size_uniformity']:<15.2f}")
    print(f"{'처리 시간(초)':<20} {fixed_stats.get('chunking_time', 'N/A'):<15} {chunking_time:<15.2f}")
except FileNotFoundError:
    print("Fixed chunking 결과를 찾을 수 없습니다. 02_fixed_chunking.ipynb를 먼저 실행하세요.")

## 8. 임베딩 및 인덱싱

In [ ]:
# 임베더 초기화
embedder = BGEEmbedder(
    model_name=config['embedding']['model_name'],
    batch_size=config['embedding']['batch_size']
)

In [ ]:
# 임베딩 생성
print("\n임베딩 생성 중...")
texts = [chunk.text for chunk in chunks]
embeddings = embedder.embed_documents(texts)

In [ ]:
# 벡터 검색기 초기화 및 인덱싱
retriever = VectorRetriever(embedder)
retriever.add_chunks(chunks, embeddings)

## 9. 테스트 검색

In [ ]:
# 테스트 쿼리 로드
test_queries = loader.load_test_queries('../data/evaluation/test_queries.json')

# 샘플 쿼리로 검색 테스트
sample_query = test_queries[0]['query']
print(f"테스트 쿼리: {sample_query}")
print("=" * 60)

results = retriever.search(sample_query, k=5)

for result in results:
    print(f"\n[Rank {result.rank}] Score: {result.score:.4f}")
    print(f"Chunk ID: {result.chunk_id}")
    print(f"Text: {result.text[:200]}...")

## 10. 전체 쿼리 검색 및 결과 저장

In [ ]:
# 전체 테스트 쿼리에 대해 검색 실행
print("전체 쿼리 검색 중...")

retrieval_results = {}

for query_data in test_queries:
    query_id = query_data['query_id']
    query_text = query_data['query']
    
    results = retriever.search(query_text, k=5)
    retrieval_results[query_id] = [r.chunk_id for r in results]

print(f"총 {len(retrieval_results)}개 쿼리 검색 완료")

In [ ]:
# 결과 저장
results_dir = Path('../results/recursive_chunking')
results_dir.mkdir(parents=True, exist_ok=True)

# 청크 저장
chunks_data = [chunk.to_dict() for chunk in chunks]
with open(results_dir / 'chunks.json', 'w', encoding='utf-8') as f:
    json.dump(chunks_data, f, ensure_ascii=False, indent=2)

# 검색 결과 저장
with open(results_dir / 'retrieval_results.json', 'w', encoding='utf-8') as f:
    json.dump(retrieval_results, f, ensure_ascii=False, indent=2)

# 통계 저장
stats['chunking_time'] = chunking_time
with open(results_dir / 'stats.json', 'w', encoding='utf-8') as f:
    json.dump(stats, f, ensure_ascii=False, indent=2)

print(f"결과 저장 완료: {results_dir}")

## 11. 다음 단계

Recursive Chunking 실험이 완료되었습니다.

다음 노트북: **04_semantic_chunking.ipynb**